# CTGAN & TVAE — Generative Oversampling

**Goal:** Εκπαιδεύουμε CTGAN και TVAE στα πραγματικά stroke samples του train set και παράγουμε synthetic stroke samples για να ισορροπήσουμε το dataset.

**Βιβλιοθήκη:** SDV (Synthetic Data Vault)  
**Σύγκριση:** Τα αποτελέσματα θα συγκριθούν με τα baselines του notebook 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, average_precision_score, matthews_corrcoef, classification_report
)

from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
from sdv.metadata import SingleTableMetadata

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

DATA_PATH = Path('../../data/healthcare-dataset-stroke-data.csv')
RANDOM_STATE = 42

## 1. Load & Split

Φορτώνουμε τα **raw data** (χωρίς encoding) γιατί το CTGAN/TVAE κάνουν το δικό τους preprocessing εσωτερικά. Κάνουμε train/test split πριν αγγίξουμε τα generative models — τα test data δεν πρέπει να επηρεάζουν την εκπαίδευση.

In [ ]:
df = pd.read_csv(DATA_PATH)

df = df.drop(columns=['id'])
df = df[df['gender'] != 'Other'].copy()
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

# Train/test split στα raw data (πριν το encoding)
train_df, test_df = train_test_split(
    df, test_size=0.2, random_state=RANDOM_STATE, stratify=df['stroke']
)

print(f'Train: {train_df.shape} | Stroke: {train_df["stroke"].sum()} ({train_df["stroke"].mean()*100:.1f}%)')
print(f'Test:  {test_df.shape}  | Stroke: {test_df["stroke"].sum()} ({test_df["stroke"].mean()*100:.1f}%)')

## 2. Minority Class — Train Set

Εκχωρούμε μόνο τα stroke=1 samples από το train set. Αυτά θα χρησιμοποιηθούν για να εκπαιδεύσουμε τα generative models.

In [ ]:
minority_train = train_df[train_df['stroke'] == 1].copy()

n_majority = (train_df['stroke'] == 0).sum()
n_minority = (train_df['stroke'] == 1).sum()
n_to_generate = n_majority - n_minority

print(f'Πραγματικά stroke samples (train): {n_minority}')
print(f'Synthetic samples που χρειαζόμαστε: {n_to_generate}')
print(f'\nΔείγμα minority class:')
minority_train.head(3)

## 3. CTGAN

**CTGAN (Conditional Tabular GAN):** GAN φτιαγμένο ειδικά για tabular data. Χρησιμοποιεί mode-specific normalization για numerical features και conditional vector για categoricals.

### 3.1 Metadata

Το SDV χρειάζεται να ξέρει τον τύπο κάθε στήλης για να κάνει σωστό preprocessing εσωτερικά.

In [ ]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(minority_train)

# Διορθώνουμε τύπους που ενδέχεται να εντοπιστούν λάθος
metadata.update_column('age',               sdtype='numerical')
metadata.update_column('avg_glucose_level', sdtype='numerical')
metadata.update_column('bmi',               sdtype='numerical')
metadata.update_column('hypertension',      sdtype='categorical')
metadata.update_column('heart_disease',     sdtype='categorical')
metadata.update_column('gender',            sdtype='categorical')
metadata.update_column('ever_married',      sdtype='categorical')
metadata.update_column('work_type',         sdtype='categorical')
metadata.update_column('Residence_type',    sdtype='categorical')
metadata.update_column('smoking_status',    sdtype='categorical')
metadata.update_column('stroke',            sdtype='categorical')

print(metadata)

### 3.2 Εκπαίδευση CTGAN

In [ ]:
ctgan = CTGANSynthesizer(
    metadata,
    epochs=300,
    verbose=True
)

ctgan.fit(minority_train)
print('\nCTGAN εκπαίδευση ολοκληρώθηκε.')

### 3.3 Παραγωγή Synthetic Samples

In [ ]:
synthetic_ctgan = ctgan.sample(num_rows=n_to_generate)
synthetic_ctgan['stroke'] = 1  # όλα τα παραγόμενα samples είναι stroke cases

print(f'Παράχθηκαν {len(synthetic_ctgan)} synthetic stroke samples')
print(f'\nΔείγμα synthetic data:')
synthetic_ctgan.head(3)

### 3.4 Quality Check — Σύγκριση Κατανομών

Ελέγχουμε αν τα synthetic data μοιάζουν με τα πραγματικά stroke samples. Αν ο CTGAN έχει μάθει σωστά, οι κατανομές πρέπει να είναι παρόμοιες.

In [ ]:
num_features = ['age', 'avg_glucose_level', 'bmi']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat in zip(axes, num_features):
    ax.hist(minority_train[feat], bins=25, alpha=0.6, label='Πραγματικά', color='#2196F3', density=True)
    ax.hist(synthetic_ctgan[feat], bins=25, alpha=0.6, label='CTGAN', color='#F44336', density=True)
    ax.set_title(feat)
    ax.legend()

plt.suptitle('CTGAN — Σύγκριση Κατανομών (Numerical)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../data/ctgan_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Σύγκριση βασικών στατιστικών
print('=== Στατιστικά Σύγκρισης ===')
for feat in num_features:
    real_mean = minority_train[feat].mean()
    synt_mean = synthetic_ctgan[feat].mean()
    real_std  = minority_train[feat].std()
    synt_std  = synthetic_ctgan[feat].std()
    print(f'{feat}:')
    print(f'  Πραγματικά: mean={real_mean:.2f}, std={real_std:.2f}')
    print(f'  CTGAN:      mean={synt_mean:.2f}, std={synt_std:.2f}')

## 4. TVAE

**TVAE (Tabular Variational Autoencoder):** VAE-based προσέγγιση. Αντί για GAN (generator vs discriminator), χρησιμοποιεί encoder-decoder αρχιτεκτονική. Γενικά πιο σταθερό στην εκπαίδευση από το GAN.

### 4.1 Εκπαίδευση TVAE

In [ ]:
tvae = TVAESynthesizer(
    metadata,
    epochs=300,
    verbose=True
)

tvae.fit(minority_train)
print('\nTVAE εκπαίδευση ολοκληρώθηκε.')

### 4.2 Παραγωγή & Quality Check

In [ ]:
synthetic_tvae = tvae.sample(num_rows=n_to_generate)
synthetic_tvae['stroke'] = 1

print(f'Παράχθηκαν {len(synthetic_tvae)} synthetic stroke samples')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat in zip(axes, num_features):
    ax.hist(minority_train[feat], bins=25, alpha=0.6, label='Πραγματικά', color='#2196F3', density=True)
    ax.hist(synthetic_tvae[feat], bins=25, alpha=0.6, label='TVAE', color='#4CAF50', density=True)
    ax.set_title(feat)
    ax.legend()

plt.suptitle('TVAE — Σύγκριση Κατανομών (Numerical)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../../data/tvae_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=== Στατιστικά Σύγκρισης ===')
for feat in num_features:
    real_mean = minority_train[feat].mean()
    synt_mean = synthetic_tvae[feat].mean()
    real_std  = minority_train[feat].std()
    synt_std  = synthetic_tvae[feat].std()
    print(f'{feat}:')
    print(f'  Πραγματικά: mean={real_mean:.2f}, std={real_std:.2f}')
    print(f'  TVAE:       mean={synt_mean:.2f}, std={synt_std:.2f}')

## 5. Augment & Classify

Για κάθε generative model:
1. Συνδυάζουμε τα πραγματικά train data με τα synthetic stroke samples
2. Κάνουμε encoding + scaling (ίδιο pipeline με notebook 2)
3. Εκπαιδεύουμε LR + RF στο augmented dataset
4. Αξιολογούμε στο **αμετάβλητο test set**

In [ ]:
NUMERICAL   = ['age', 'avg_glucose_level', 'bmi']
CATEGORICAL = ['hypertension', 'heart_disease', 'gender', 'ever_married',
               'work_type', 'Residence_type', 'smoking_status']

def build_Xy(df_raw):
    """Encode + prepare feature matrix from raw dataframe."""
    d = df_raw.copy()
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    d[CATEGORICAL] = enc.fit_transform(d[CATEGORICAL])
    X = d[NUMERICAL + CATEGORICAL].values
    y = d['stroke'].values
    return X, y, enc

def scale_split(X_tr, X_te):
    """StandardScaler on numerical columns only."""
    scaler = StandardScaler()
    n = len(NUMERICAL)
    X_tr_s = np.hstack([scaler.fit_transform(X_tr[:, :n]), X_tr[:, n:]])
    X_te_s = np.hstack([scaler.transform(X_te[:, :n]),     X_te[:, n:]])
    return X_tr_s, X_te_s

results = []

def evaluate(method, clf_name, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]
    f1     = f1_score(y_te, y_pred)
    pr_auc = average_precision_score(y_te, y_prob)
    mcc    = matthews_corrcoef(y_te, y_pred)
    print(f'\n[{method}] {clf_name}')
    print(f'  F1: {f1:.4f} | PR-AUC: {pr_auc:.4f} | MCC: {mcc:.4f}')
    print(classification_report(y_te, y_pred, target_names=['No Stroke', 'Stroke']))
    results.append({'Method': method, 'Classifier': clf_name,
                    'F1': round(f1,4), 'PR-AUC': round(pr_auc,4), 'MCC': round(mcc,4)})

# Encode test set (fit encoder on train να αποφύγουμε leakage)
X_train_raw, y_train_raw, enc_train = build_Xy(train_df)
X_test_raw  = enc_train.transform(test_df[CATEGORICAL].values)
X_test_full = np.hstack([test_df[NUMERICAL].values, X_test_raw])
y_test      = test_df['stroke'].values

In [ ]:
# --- CTGAN Augmentation ---
train_ctgan = pd.concat([train_df, synthetic_ctgan], ignore_index=True)

X_tr_ctgan, y_tr_ctgan, enc_ctgan = build_Xy(train_ctgan)
X_te_ctgan = np.hstack([test_df[NUMERICAL].values,
                         enc_ctgan.transform(test_df[CATEGORICAL].values)])

X_tr_ctgan_s, X_te_ctgan_s = scale_split(X_tr_ctgan, X_te_ctgan)

print(f'Augmented train (CTGAN): {train_ctgan.shape} | Stroke: {y_tr_ctgan.sum()} ({y_tr_ctgan.mean()*100:.1f}%)')

evaluate('CTGAN', 'Logistic Regression',
         LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
         X_tr_ctgan_s, y_tr_ctgan, X_te_ctgan_s, y_test)

evaluate('CTGAN', 'Random Forest',
         RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
         X_tr_ctgan_s, y_tr_ctgan, X_te_ctgan_s, y_test)

In [ ]:
# --- TVAE Augmentation ---
train_tvae = pd.concat([train_df, synthetic_tvae], ignore_index=True)

X_tr_tvae, y_tr_tvae, enc_tvae = build_Xy(train_tvae)
X_te_tvae = np.hstack([test_df[NUMERICAL].values,
                        enc_tvae.transform(test_df[CATEGORICAL].values)])

X_tr_tvae_s, X_te_tvae_s = scale_split(X_tr_tvae, X_te_tvae)

print(f'Augmented train (TVAE): {train_tvae.shape} | Stroke: {y_tr_tvae.sum()} ({y_tr_tvae.mean()*100:.1f}%)')

evaluate('TVAE', 'Logistic Regression',
         LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
         X_tr_tvae_s, y_tr_tvae, X_te_tvae_s, y_test)

evaluate('TVAE', 'Random Forest',
         RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
         X_tr_tvae_s, y_tr_tvae, X_te_tvae_s, y_test)

## 6. Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

In [ ]:
metrics = ['F1', 'PR-AUC', 'MCC']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = {'Logistic Regression': '#2196F3', 'Random Forest': '#F44336'}

for ax, metric in zip(axes, metrics):
    for clf_name, grp in results_df.groupby('Classifier'):
        x = np.arange(len(grp))
        ax.bar(
            grp['Method'].values,
            grp[metric].values,
            label=clf_name,
            alpha=0.8,
            color=colors[clf_name]
        )
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=10)
    ax.legend(fontsize=8)

plt.suptitle('CTGAN vs TVAE', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../../data/ctgan_tvae_results.png', dpi=150, bbox_inches='tight')
plt.show()